# Hybrid Retrieval — Amazon Product Dataset 2020

Covers task backlog §1.4, reading the collections built in
[1.3 build index.ipynb](1.3%20build%20index.ipynb) (no rebuild here — just search):

- vector search + metadata filters combined (budget, category; brand as a post-filter — see why below)
- unit-aware tie-breaking (price-per-unit) among near-equally-relevant results
- results carry `doc_id` for citation traceability
- reranker skipped for now (backlog marks it optional; revisit in §1.5 if eval numbers call for it)

No new dependencies — reads the existing Chroma collections with the same
`sentence-transformers` model used to build them.

In [1]:
# !pip install pandas sentence-transformers chromadb python-dotenv

## Setup & config

In [2]:
import os

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

from dotenv import load_dotenv

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
load_dotenv(os.path.join(REPO_ROOT, ".env"))

EMBEDDING_MODEL = os.environ.get("EMBEDDING_MODEL", "all-MiniLM-L6-v2")
CHROMA_PERSIST_DIR_HK = os.path.join(REPO_ROOT, os.environ.get(
    "CHROMA_PERSIST_DIR_HOME_KITCHEN", "Code/1. Data Pipeline & Indexing/processed/chroma_db_home_kitchen"
))
CHROMA_PERSIST_DIR_ALL = os.path.join(REPO_ROOT, os.environ.get(
    "CHROMA_PERSIST_DIR_ALL", "Code/1. Data Pipeline & Indexing/processed/chroma_db_all"
))
CHROMA_COLLECTION_HK = os.environ.get("CHROMA_COLLECTION", "products_home_kitchen")
CHROMA_COLLECTION_ALL = os.environ.get("CHROMA_COLLECTION_ALL", "products_all")

import chromadb
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(EMBEDDING_MODEL)
client_hk = chromadb.PersistentClient(path=CHROMA_PERSIST_DIR_HK)
client_all = chromadb.PersistentClient(path=CHROMA_PERSIST_DIR_ALL)
collection_hk = client_hk.get_collection(CHROMA_COLLECTION_HK)
collection_all = client_all.get_collection(CHROMA_COLLECTION_ALL)

print(f"{CHROMA_COLLECTION_HK}: {collection_hk.count():,} products")
print(f"{CHROMA_COLLECTION_ALL}: {collection_all.count():,} products")

products_home_kitchen: 706 products
products_all: 9,930 products


## The `search()` function

**Category and budget filters go through Chroma's `where` clause** — reliable, since
`category_top` is a clean exact string and `price` is a clean float, both filterable natively
(`$eq`, `$gte`, `$lte`).

**Brand does not** — recall from §1.2 that `brand` is a low-confidence regex-derived field
(exact match only ~5/12 in the hand-reviewed sample), so filtering it as an *exact* metadata
match would silently drop good results over extraction noise (e.g. `"Wildkin Kids"` vs a user
typing `"Wildkin"`). Instead, brand filtering **over-fetches** candidates by vector search
first, then does a case-insensitive substring match in Python — forgiving of the exact
extraction, at the cost of not being usable as a hard pre-filter.

In [3]:
def search(query, collection=None, top_k=5, budget_min=None, budget_max=None,
           category=None, brand=None, overfetch=30):
    """
    Hybrid retrieval: vector similarity + metadata filters, with unit-aware
    tie-breaking. Returns a list of dicts, each carrying `doc_id` for citation.
    """
    collection = collection or collection_hk

    where_clauses = []
    if budget_min is not None:
        where_clauses.append({"price": {"$gte": budget_min}})
    if budget_max is not None:
        where_clauses.append({"price": {"$lte": budget_max}})
    if category is not None:
        where_clauses.append({"category_top": category})
    where = None
    if len(where_clauses) == 1:
        where = where_clauses[0]
    elif len(where_clauses) > 1:
        where = {"$and": where_clauses}

    query_embedding = model.encode([query])[0].tolist()
    n_results = overfetch if brand else top_k
    results = collection.query(query_embeddings=[query_embedding], n_results=n_results, where=where)

    candidates = []
    for meta, dist, doc in zip(results["metadatas"][0], results["distances"][0], results["documents"][0]):
        if brand and brand.lower() not in meta.get("brand", "").lower():
            continue
        candidates.append({
            "doc_id": meta["id"],
            "title": meta.get("title", doc),
            "price": meta.get("price"),
            "price_per_unit": meta.get("price_per_unit"),
            "price_unit": meta.get("price_unit"),
            "brand": meta.get("brand"),
            "category_top": meta.get("category_top"),
            "url": meta.get("url"),
            "distance": dist,
        })

    candidates = _unit_aware_tie_break(candidates)
    return candidates[:top_k]


def _unit_aware_tie_break(candidates, group_tolerance=0.15):
    """
    Groups candidates by price_unit (e.g. all the $/ct items). Within a group,
    if that group's OWN distance spread is <= group_tolerance -- i.e. those
    specific items are near-substitutes of each other, regardless of how far
    they are from the single best match -- re-sort just that group by
    price-per-unit (cheapest first), swapped into the positions the group
    already occupied. Doesn't change overall relevance ranking, just reorders
    genuine same-unit substitutes relative to each other.
    """
    if not candidates:
        return candidates
    result = list(candidates)

    by_unit = {}
    for i, c in enumerate(result):
        if c["price_per_unit"] is not None and c["price_unit"] is not None:
            by_unit.setdefault(c["price_unit"], []).append(i)

    for unit, idxs in by_unit.items():
        if len(idxs) < 2:
            continue
        group = [result[i] for i in idxs]
        spread = max(c["distance"] for c in group) - min(c["distance"] for c in group)
        if spread <= group_tolerance:
            for i, c in zip(idxs, sorted(group, key=lambda c: c["price_per_unit"])):
                result[i] = c
    return result

## Try it — category + budget filter

Real query against the real index, read the actual results.

In [4]:
results = search("lunch box", budget_max=15, top_k=5)
for r in results:
    print(f"[{r['distance']:.3f}] {r['title'][:65]}  (${r['price']:.2f})  doc_id={r['doc_id'][:8]}...")

[0.698] LEGO Lunch Box, Medium Pink  ($11.29)  doc_id=42d621ee...
[0.751] Wildkin 33600 Monsters Lunch Box, Insulated, Moisture Resistant,   ($14.99)  doc_id=b1904b27...
[0.772] Stephen Joseph All Over Print Lunch Box, Owl  ($14.00)  doc_id=adf1d218...
[0.797] Wildkin Kids Insulated Lunch Box for Boys and Girls, Perfect Size  ($12.77)  doc_id=7c3f2bf5...
[0.841] Wildkin Kids Insulated Lunch Box for Boys and Girls, Perfect Size  ($10.88)  doc_id=5d953e35...


## Try it — brand filter (post-filter, not a hard metadata match)

In [5]:
results = search("insulated lunch box", brand="Wildkin", top_k=5)
for r in results:
    print(f"[{r['distance']:.3f}] {r['title'][:65]}  brand={r['brand']!r}  (${r['price']:.2f})")

[0.626] Wildkin Kids Insulated Lunch Box for Boys and Girls, Perfect Size  brand='Wildkin Kids'  ($12.77)
[0.644] Wildkin 33600 Monsters Lunch Box, Insulated, Moisture Resistant,   brand='Wildkin'  ($14.99)
[0.690] Wildkin Kids Insulated Lunch Box for Boys and Girls, Perfect Size  brand='Wildkin Kids'  ($10.88)
[1.026] Wildkin 33527 Rip-Stop Black/Green Box Lunch Bags, One Size  brand='Wildkin'  ($12.59)
[1.162] Wildkin 33417 Olive Kids Fairy Princess Lunch Box  brand='Wildkin'  ($13.41)


## Try it — full catalog collection, not just the demo slice

Same function, just pass `collection_all` — this is what the "configurable, defaults to
Home & Kitchen" scope decision buys: no rewrite needed to search the whole catalog.

In [6]:
results = search("remote control car", collection=collection_all, top_k=5)
for r in results:
    print(f"[{r['distance']:.3f}] {r['title'][:65]}  ({r['category_top']})  (${r['price']:.2f})")

[0.983] Kid Galaxy Remote Control Bumper Cars. RC 2 Player Game. 2 Cars a  (Toys & Games)  ($59.99)
[1.008] Dimple Red RC Light Up Rechargeable Remote Controlled Acrobatic F  (Toys & Games)  ($14.98)
[1.015] Carmel 1:14 Ferrari 458 Italia 2.4Ghz with Lights Rechargeable Ba  (Home & Kitchen)  ($56.51)
[1.074] Maisto R/C 1:24 Scale 2017 Ford GT NAIAS Variable Color Radio Con  (Toys & Games)  ($19.03)
[1.074] Hey! Play! 80-YF-141F147 Remote Control Front Loader- 1: 24 Scale  (Toys & Games)  ($22.99)


## Unit-aware tie-breaking — does it actually trigger?

Groups candidates by `price_unit`; within a group, if the items' own distance spread is tight enough to be genuine substitutes, re-sorts that group by price-per-unit. Confirmed below with a real query, not assumed.

In [7]:
results = search("plastic party plates", top_k=8)
for r in results:
    ppu = f"${r['price_per_unit']:.3f}/{r['price_unit']}" if r["price_per_unit"] else "n/a"
    print(f"[{r['distance']:.3f}] {r['title'][:55]}  price/unit={ppu}")

[0.603] Party Essentials 40-Count Hard Plastic 6" Dessert/Appet  price/unit=n/a
[0.885] Amscan Dream Big Disposable Paper Dessert Plates Gradua  price/unit=n/a
[1.002] Marvel Epic AvengersTM Square Plates, 9", Party Supply,  price/unit=$0.855/ct
[0.937] 8-Count Large Round Paper Banquet Plates, Ladybug Fancy  price/unit=n/a
[0.977] Party Essentials N15090 Plastic Shot Glasses, 50-Count,  price/unit=n/a
[0.997] Party Essentials Hard Plastic Two Piece 12-Ounce Margar  price/unit=$1.071/ct
[0.922] Party Essentials Hard Plastic Two Piece 5-Ounce Champag  price/unit=$1.089/ct
[1.008] Golden Wedding Dessert Plates 8ct | Wedding and Engagem  price/unit=n/a


## Conclusion

`search()` combines vector similarity with filters (category, budget, brand) and works against either index. Next: [1.5 rag evaluation.ipynb](1.5%20rag%20evaluation.ipynb) checks how good the results actually are.